<a href="https://colab.research.google.com/github/contreras-juan/Material-Ciencia-de-Datos/blob/main/Deep_Learning/Ejercicios/04_Taller_Autoencoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<h1 style="color: #FECB05; text-align: center;">Taller práctico: Autoencoders</h1>


<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)


---


<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica autoencoders con **datos y ejercicios distintos** a los del cuaderno `9_Autoencoders.ipynb`.
Aquí no reutilizamos MNIST, el denoising gaussiano ni la detección 1 vs 7 de las notas.

Enfocamos:

1. Formas, MSE de reconstrucción y el cuello de botella
2. Detección de **fuga de información** al entrenar un detector de anomalías
3. Autoencoder denso en **Fashion-MNIST** y efecto de $\dim(z)$
4. Denoising con ruido **sal y pimienta**
5. Anomalías en **ECG** (señales reales 1D)
6. Autoencoder **convolucional** en Fashion-MNIST
7. El encoder como **extractor de características**
8. Flujo completo en un dataset **tabular** (cáncer de mama)

**Cómo trabajar:**

- Completa las celdas con `# Escribe tu código aquí`.
- El objetivo de un autoencoder es la **propia entrada**, no una etiqueta de clase.
- En anomalías: entrena **solo** con la clase normal; el umbral sale de un **val normal**.
- Escala con estadísticas **solo de train** (o solo de la clase normal de train).
- Reporta MSE / AUC y, cuando aplique, reconstrucciones o histogramas de error.

**Tiempo sugerido:** 2.5–3.5 horas.


<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [Ejercicio 1. Formas y pérdida de reconstrucción](#ejercicio-1)
- [Ejercicio 2. Cacería de leakage](#ejercicio-2)
- [Ejercicio 3. Autoencoder denso en Fashion-MNIST](#ejercicio-3)
- [Ejercicio 4. Denoising sal y pimienta](#ejercicio-4)
- [Ejercicio 5. Anomalías en ECG](#ejercicio-5)
- [Ejercicio 6. Autoencoder convolucional](#ejercicio-6)
- [Ejercicio 7. Encoder como extractor](#ejercicio-7)
- [Ejercicio integrador. Cáncer de mama](#ejercicio-integrador)


---


<h2 style="color: #007ACC;">Instalación e importaciones</h2>

Ejecuta primero la celda de instalación y luego la de importaciones.


In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import (
    Conv2D,
    Conv2DTranspose,
    Dense,
    Input,
    MaxPooling2D,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


In [ ]:
def plot_history(history, title="Curvas de entrenamiento"):
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, hist["loss"], label="train loss")
    if "val_loss" in hist:
        plt.plot(epochs, hist["val_loss"], label="val loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def show_images(rows, titles, n=10, h=28, w=28, cmap="gray"):
    """Una fila por arreglo; `n` columnas."""
    n_rows = len(rows)
    plt.figure(figsize=(20, 2.2 * n_rows))
    for r, (imgs, title) in enumerate(zip(rows, titles)):
        for i in range(n):
            ax = plt.subplot(n_rows, n, r * n + i + 1)
            plt.imshow(np.asarray(imgs[i]).reshape(h, w), cmap=cmap)
            if i == 0:
                ax.set_ylabel(title, fontsize=11)
            plt.xticks([])
            plt.yticks([])
    plt.tight_layout()
    plt.show()


FASHION_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]


---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Formas y pérdida de reconstrucción</h2>

**Objetivo:** fijar la métrica y la geometría del cuello de botella antes de entrenar redes.


1. Implementa `reconstruction_mse(x_true, x_hat)` que devuelva **un error por muestra** (vector de longitud $N$), aplanando si hace falta. Comprueba:
   - si `x_hat = x_true`, la media del vector es $0$;
   - con `x_true = [[1, 0, 0], [0, 1, 0]]` y `x_hat = [[1, 0, 0], [1, 1, 0]]`, el segundo error es $1/3$ (MSE, no suma de cuadrados).
2. Implementa `linear_bottleneck(X, k)`: centra $X$ (shape `(N, d)`), calcula el SVD económico y reconstruye usando solo $k$ componentes (equivalente a un autoencoder **lineal** undercomplete). Prueba con $N=20$, $d=6$, $k=2$ y compara `reconstruction_mse` para $k=2$ vs $k=6$.
3. Responde: si $\dim(z)=\dim(x)$ y encoder/decoder son lineales, ¿qué puede aprender la red? ¿Por qué en clase usamos $\dim(z) \ll \dim(x)$ o ruido?


In [ ]:
# Escribe tu código aquí
def reconstruction_mse(x_true, x_hat):
    pass


def linear_bottleneck(X, k):
    pass


# Pruebas y respuesta:


---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Cacería de leakage</h2>

**Objetivo:** detectar y corregir el pipeline típico (y incorrecto) de un autoencoder para anomalías.


Datos sintéticos 2D: normales $\sim \mathcal{N}(0, I)$ y anomalías $\sim \mathcal{N}(5, I)$.

El siguiente pipeline tiene **al menos 3 errores**. No lo copies: **reescríbelo bien**.

```python
# PIPELINE INCORRECTO (solo para análisis)
np.random.seed(0)
normales = np.random.normal(0, 1, (800, 2))
anomalos = np.random.normal(5, 1, (80, 2))
X = np.vstack([normales, anomalos])
y = np.array([0] * 800 + [1] * 80)          # 1 = anomalía

scaler = MinMaxScaler()
X = scaler.fit_transform(X)                 # (1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0    # (2)
)

# autoencoder.fit(X_train, X_train)         # ve anomalías en train
recon = autoencoder.predict(X_test)
err = np.mean((recon - X_test) ** 2, axis=1)
umbral = np.percentile(err, 95)             # (3)
```

1. Enumera los errores (1), (2) y (3) y el daño de cada uno.
2. Implementa la versión correcta:
   - split de **normales** en train/val/test;
   - scaler **solo** con train normal;
   - AE entrenado **solo** con train normal;
   - umbral = percentil 95 del error en **val normal**;
   - evalúa TPR en anomalías de test y FPR en normales de test.
3. Grafica el scatter 2D de test (normal vs anómalo predicho) y el histograma de errores.


In [ ]:
# Escribe tu código aquí
# Errores:
# (1)
# (2)
# (3)

rng = np.random.default_rng(SEED)
normales = rng.normal(0, 1, size=(800, 2))
anomalos = rng.normal(5, 1, size=(80, 2))



---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Autoencoder denso en Fashion-MNIST</h2>

**Objetivo:** entrenar un AE undercomplete en prendas (no dígitos) y medir el efecto de $\dim(z)$.


Fashion-MNIST se carga con `fashion_mnist.load_data()`: imágenes $28\times 28$ de 10 tipos de ropa. Las **etiquetas no se usan** para entrenar el autoencoder.

1. Normaliza a $[0, 1]$ y aplana a `(N, 784)`.
2. Construye encoder/decoder que **compartan pesos** (nombra la capa latente). Latente **lineal**. Salida `sigmoid`. Pérdida MSE.
3. Entrena **dos** modelos con el mismo presupuesto (`EarlyStopping` + `ReduceLROnPlateau`, p. ej. 12 épocas, batch 256):
   - Modelo A: `latent_dim=8`
   - Modelo B: `latent_dim=64`
4. Reporta MSE de test de ambos y muestra 10 reconstrucciones de cada uno.
5. **Pregunta:** ¿el de menor MSE es siempre el mejor *como representación*? Relaciona con el riesgo de aprender (casi) la identidad.


In [ ]:
# Escribe tu código aquí
(x_train_img, y_train), (x_test_img, y_test) = fashion_mnist.load_data()



---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Denoising sal y pimienta</h2>

**Objetivo:** denoising con un ruido **distinto** al gaussiano de las notas de clase.


Ruido **sal y pimienta:** con probabilidad $p/2$ un píxel pasa a $0$; con probabilidad $p/2$ pasa a $1$; el resto no cambia. Usa $p=0.3$ y `rng` con `SEED`.

1. Implementa `salt_pepper(X, p, rng)` sobre arreglos aplanados en $[0, 1]$.
2. Entrena un AE denso (`latent_dim=32`) con entrada ruidosa y **objetivo limpio**.
3. En test, muestra original / ruidosa / reconstruida y reporta MSE respecto a la imagen limpia.
4. **Pregunta:** ¿por qué este ruido impide que la red copie píxeles, incluso con un latente holgado?


In [ ]:
# Escribe tu código aquí
def salt_pepper(X, p=0.3, rng=None):
    pass



---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Anomalías en ECG</h2>

**Objetivo:** detector de anomalías en series reales 1D (no imágenes).


Dataset (mismo que el tutorial de TensorFlow, **no** está en las notas del curso):

`https://storage.googleapis.com/download.tensorflow.org/data/ecg.csv`

- $4998$ latidos, $140$ muestras cada uno.
- Última columna: etiqueta. **$1$ = ritmo normal**, **$0$ = anómalo**.

1. Separa features y etiqueta. Split **estratificado** 80/20 para tener normales y anómalos en test.
2. `MinMaxScaler` ajustado **solo** con los latidos **normales del train**.
3. Entrena un AE denso solo con normales de train (`latent_dim` pequeño, p. ej. $8$–$16$).
4. Umbral = percentil $95$ del MSE en un **val normal** (parte el train normal).
5. Reporta FPR en normales de test, TPR en anómalos de test y, si puedes, **AUC** del error de reconstrucción (`roc_auc_score`).
6. Grafica un latido normal bien reconstruido y uno anómalo mal reconstruido.


In [ ]:
# Escribe tu código aquí
ECG_URL = "https://storage.googleapis.com/download.tensorflow.org/data/ecg.csv"



---

<a id="ejercicio-6"></a>
<h2 style="color: #007ACC;">Ejercicio 6. Autoencoder convolucional</h2>

**Objetivo:** no aplanar Fashion-MNIST; usar geometría 2D.


1. Forma `(N, 28, 28, 1)`.
2. Encoder: `Conv2D` + `MaxPooling2D` hasta un mapa latente (p. ej. $7\times 7\times 8$).
3. Decoder: `Conv2DTranspose` con `strides=2` y `Conv2D` final `sigmoid`.
4. Entrena con el mismo tipo de callbacks que el ejercicio 3.
5. Compara MSE de test con el **mejor** AE denso del ejercicio 3.
6. **Pregunta:** en $28\times 28$ en escala de grises la diferencia puede ser pequeña. ¿En qué régimen esperarías que gane el convolucional?


In [ ]:
# Escribe tu código aquí



---

<a id="ejercicio-7"></a>
<h2 style="color: #007ACC;">Ejercicio 7. Encoder como extractor</h2>

**Objetivo:** usar $z$ en una tarea supervisada **sin** reentrenar el autoencoder.


Toma el encoder del modelo B del ejercicio 3 (`latent_dim=64`), o vuélvelo a entrenar si no lo tienes en memoria.

1. Obtén $z_{\text{train}}$ y $z_{\text{test}}$ con `encoder.predict` (**sin** usar `y` en el AE).
2. Entrena `LogisticRegression` (o un `Dense` lineal) para las $10$ clases:
   - sonda A: sobre $z$;
   - sonda B: sobre píxeles aplanados (puedes submuestrear train a $10\,000$ ejemplos si B es lento).
3. Reporta exactitud de test de A y B, y $\dim(z)$ vs $784$.
4. Opcional: t-SNE de $z_{\text{test}}$ (máx. $3000$ puntos) coloreado por clase.
5. **Pregunta:** el AE no vio etiquetas. Si A se acerca a B, ¿qué implica sobre $z$?


In [ ]:
# Escribe tu código aquí



---

<a id="ejercicio-integrador"></a>
<h2 style="color: #007ACC;">Ejercicio integrador. Cáncer de mama</h2>

**Objetivo:** autoencoder tabular + anomalías, con el mismo rigor de leakage que el ejercicio 2.


`sklearn.datasets.load_breast_cancer`: $30$ features clínicas. En scikit-learn, **$0$ = maligno**, **$1$ = benigno**. Trata **benigno como normal** y **maligno como anomalía**.

1. Split estratificado. A partir del train, quédate con los benignos para ajustar el scaler (`StandardScaler`) y entrenar el AE.
2. Decoder **lineal** (las features no están en $[0, 1]$). Pérdida MSE. `latent_dim` $\approx 8$.
3. Umbral desde el error de un val **benigno**.
4. Reporta FPR (benignos de test), TPR (malignos de test) y AUC del error.
5. Compara contra un baseline tonto: umbral sobre la norma $\lVert x \rVert$ (sin AE). ¿Gana el autoencoder?
6. Párrafo de conclusión: generalización, leakage evitado y límite de un AE pequeño en datos tabulares.

Entrega esperada: código reproducible, histograma de errores, métricas impresas y un párrafo.


In [ ]:
# Escribe tu código aquí
# Experimento integrador (breast cancer)

